# 02 — Object References, Copying, Garbage Collection, and Weakrefs

Explore references, aliases, deep copies, cycles, and weak references without assuming every Python implementation manages memory exactly like CPython.

*Advanced · 90–150 min*

## Objectives
- Distinguish binding, aliasing, and mutation
- Compare shallow and deep copying
- Describe cycles, garbage collection, and deterministic cleanup
- Use weak references safely

## Examples

### 1. Aliasing versus rebinding

Assigning a list to another name shares the same object. A shallow copy creates a new outer container while retaining references to inner values; this can lead to surprising changes in nested structures.

In [ ]:
original = [{"score": 1}]
alias = original
shallow = original.copy()
alias[0]["score"] = 8
print(original, shallow)
shallow.append({"score": 2})
print(len(original), len(shallow))

**Design question:** Why is the inner dictionary shared after shallow copying?

### 2. Deep-copy semantics

copy.deepcopy recursively copies ordinary containers while maintaining a memo for repeated references. Do not deep-copy live file handles, locks, or other resource-owning objects as a generic solution.

In [ ]:
import copy
records = {"metadata": {"name": "sample"}, "scores": [1, 2]}
shallow = copy.copy(records)
deep = copy.deepcopy(records)
records["scores"].append(3)
print(shallow["scores"], deep["scores"])
assert deep["scores"] == [1, 2]

**Design question:** When should you construct a fresh resource instead of copying?

### 3. Cycles and lifetime

Reference cycles can prevent trivial reference-count cleanup. CPython uses reference counts plus cyclic garbage collection, but memory management varies between runtimes. Cleanup for files and sockets should use explicit close or context managers.

In [ ]:
import gc
class Node:
    def __init__(self, name):
        self.name, self.next = name, None

a, b = Node("a"), Node("b")
a.next, b.next = b, a
print("GC enabled:", gc.isenabled())
a.next = None
b.next = None
assert a.next is None and b.next is None

**Design question:** What does a deterministic resource close provide that GC cannot guarantee?

### 4. Weak references

weakref.ref does not extend a target's lifetime. Weak references are useful for observers and caches; the target may disappear once strong references are gone. Always handle None when dereferencing.

In [ ]:
import weakref
import gc
class Payload:
    pass

item = Payload()
reference = weakref.ref(item)
print(reference() is item)
del item
gc.collect()
print(reference() is None)

**Design question:** Why is weakref to a built-in list not generally available?

## Exercises

**Exercise 1:** Demonstrate shallow vs deep copying for nested dicts.

In [ ]:
# Exercise 1
# TODO: Implement, demonstrate, and add assertions.

**Exercise 2:** Build a weak-key dictionary for an object metadata cache.

In [ ]:
# Exercise 2
# TODO: Implement, demonstrate, and add assertions.

**Exercise 3:** Create and break a reference cycle, explaining when GC matters.

In [ ]:
# Exercise 3
# TODO: Implement, demonstrate, and add assertions.

**Exercise 4:** Design a context manager for a resource that should not rely on object finalization.

In [ ]:
# Exercise 4
# TODO: Implement, demonstrate, and add assertions.

## Common mistakes

- `del` removes a binding, not necessarily an object.
- Deepcopy is not a safe strategy for every resource-owning object.
- Do not assume immediate GC or finalization across Python implementations.

**Next:** [03 — Generics, TypeVars, Overloads, and Protocols](03_advanced_typing.ipynb)